# ETF Portfolio Optimiser

This Google Colab notebook runs a simple ETF portfolio optimisation workflow end-to-end.

It expects a CSV file in Google Drive with these columns:

- `ticker`
- `name`

The notebook resolves Yahoo Finance symbols, downloads 5 years of daily close prices, computes risk metrics, builds a PCA-denoised covariance matrix, and optimises long-only portfolios for both the full 5-year window and the last 252 trading days.


## 1. Install dependencies

In [ ]:
%pip -q install yfinance requests beautifulsoup4

## 2. Mount Google Drive and configure paths

The notebook creates `MyDrive/etf-optimiser/investengine_etfs.csv` by scraping InvestEngine, then loads that CSV as the optimiser input. All generated files are saved in the same Google Drive project folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/etf-optimiser')
INPUT_CSV = PROJECT_DIR / 'investengine_etfs.csv'
SYMBOL_CACHE_JSON = PROJECT_DIR / 'symbol_cache.json'
PRICES_CSV = PROJECT_DIR / 'prices_long.csv'

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project folder: {PROJECT_DIR}')
print(f'InvestEngine ETF CSV: {INPUT_CSV}')

## 3. Settings

In [ ]:
YAHOO_SUFFIXES = ['', '.L', '.DE', '.MI', '.AS', '.PA', '.SW', '.IR', '.TO', '.AX']
PCA_COMPONENTS = 5
MAX_WEIGHT = 0.30
TRADING_DAYS = 252
RISK_FREE_RATE = 0.0
TARGET_RETURN_STEPS = 50

## 4. Import libraries

In [ ]:
import json
import re
import warnings
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse


import numpy as np
import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

## 5. Scrape InvestEngine ETF universe and create the input CSV

This cell scrapes InvestEngine's public ETF range page and sitemap, extracts ETF detail page URLs, and writes a handy Drive CSV at `MyDrive/etf-optimiser/investengine_etfs.csv` with the required `ticker,name` columns.

To keep Colab responsive, the scraper does **not** fetch every ETF detail page by default. Fetching hundreds of detail pages sequentially is the step that can make Colab appear to hang or time out. If you need fuller names, set `SCRAPE_DETAIL_PAGES = True` and optionally increase `SCRAPE_MAX_DETAIL_PAGES`.


In [ ]:
INVESTENGINE_ALL_ETFS_URL = 'https://investengine.com/etfs/all/'
INVESTENGINE_SITEMAP_URL = 'https://investengine.com/sitemap.xml'
INVESTENGINE_EXCLUDED_SECTIONS = {
    'all',
    'collections',
    'industries-and-sectors',
    'independent',
    'inline-collections',
}
SCRAPE_HEADERS = {'User-Agent': 'Mozilla/5.0 ETF optimiser notebook'}
SCRAPE_REQUEST_TIMEOUT = 10
SCRAPE_DETAIL_PAGES = False
SCRAPE_MAX_DETAIL_PAGES = 25


def clean_text(value):
    return re.sub(r'\s+', ' ', value or '').strip()


def fetch_text(url, timeout=SCRAPE_REQUEST_TIMEOUT):
    response = requests.get(url, headers=SCRAPE_HEADERS, timeout=timeout)
    response.raise_for_status()
    return response.text


def etf_ticker_from_url(url):
    parts = [part for part in urlparse(url).path.split('/') if part]
    if len(parts) != 3 or parts[0] != 'etfs' or parts[1] in INVESTENGINE_EXCLUDED_SECTIONS:
        return None
    return parts[2].upper()


def name_from_card_text(text, ticker):
    text = clean_text(text)
    text = re.split(r'\bTER\b|\bYield\b|\bAccumulating\b|\bDistributing\b', text)[0]
    match = re.search(rf'(.+?)\b{re.escape(ticker)}\b', text)
    if match:
        return clean_text(match.group(1))
    return ''


def name_from_detail_page(html, ticker):
    soup = BeautifulSoup(html, 'html.parser')
    title = clean_text(soup.title.get_text(' ')) if soup.title else ''
    match = re.match(rf'(.+?)\s*\({re.escape(ticker)}\)\s*\|\s*InvestEngine', title)
    if match:
        return clean_text(match.group(1))

    heading = soup.find('h1')
    if heading:
        return clean_text(heading.get_text(' ')).replace(f' ({ticker})', '')
    return ''


def parse_etf_links_from_page(html, page_url):
    soup = BeautifulSoup(html, 'html.parser')
    rows = {}
    for link in soup.find_all('a', href=True):
        url = urljoin(page_url, link['href'])
        ticker = etf_ticker_from_url(url)
        if not ticker:
            continue
        rows[url] = {
            'ticker': ticker,
            'name': name_from_card_text(link.get_text(' '), ticker),
            'source_url': url,
        }
    return rows


def collect_sitemap_urls(url, seen=None):
    seen = seen or set()
    if url in seen:
        return set()
    seen.add(url)

    sitemap_text = fetch_text(url)
    urls = set()
    for loc_url in re.findall(r'<loc>(.*?)</loc>', sitemap_text):
        loc_url = clean_text(loc_url)
        if loc_url.endswith('.xml'):
            urls |= collect_sitemap_urls(loc_url, seen)
        else:
            urls.add(loc_url)
    return urls


def maybe_fetch_missing_names(rows_by_url):
    missing_name_items = [item for item in sorted(rows_by_url.items()) if not item[1]['name']]
    if not SCRAPE_DETAIL_PAGES:
        print(
            f'Skipping {len(missing_name_items)} ETF detail-page fetches; '
            'names missing from listing/sitemap will fall back to ticker symbols.'
        )
        return

    if SCRAPE_MAX_DETAIL_PAGES is not None:
        missing_name_items = missing_name_items[:SCRAPE_MAX_DETAIL_PAGES]
        print(f'Fetching up to {len(missing_name_items)} ETF detail pages for missing names')

    for index, (url, row) in enumerate(missing_name_items, start=1):
        try:
            row['name'] = name_from_detail_page(fetch_text(url), row['ticker'])
        except Exception as exc:
            print(f"Could not fetch ETF detail page for {row['ticker']}: {exc}")
        if index % 10 == 0:
            print(f'Checked {index} ETF detail pages...')


def scrape_investengine_etfs():
    print(f'Scraping {INVESTENGINE_ALL_ETFS_URL}', flush=True)
    all_page_html = fetch_text(INVESTENGINE_ALL_ETFS_URL)
    rows_by_url = parse_etf_links_from_page(all_page_html, INVESTENGINE_ALL_ETFS_URL)
    print(f'Found {len(rows_by_url)} ETF links on the range page', flush=True)

    try:
        sitemap_urls = collect_sitemap_urls(INVESTENGINE_SITEMAP_URL)
        etf_detail_urls = {url for url in sitemap_urls if etf_ticker_from_url(url)}
        print(f'Found {len(etf_detail_urls)} ETF detail URLs in the sitemap', flush=True)
        for url in etf_detail_urls:
            rows_by_url.setdefault(url, {
                'ticker': etf_ticker_from_url(url),
                'name': '',
                'source_url': url,
            })
    except Exception as exc:
        print(f'Sitemap scrape failed, using ETF range page links only: {exc}', flush=True)

    maybe_fetch_missing_names(rows_by_url)

    rows = [row for _, row in sorted(rows_by_url.items(), key=lambda item: item[1]['ticker'])]
    if not rows:
        raise ValueError('No InvestEngine ETFs were found. Check the page structure or network access.')

    etf_csv = pd.DataFrame(rows)
    etf_csv['name'] = etf_csv['name'].replace('', pd.NA).fillna(etf_csv['ticker'])
    etf_csv = etf_csv[['ticker', 'name']].drop_duplicates().sort_values('ticker')
    etf_csv.to_csv(INPUT_CSV, index=False)
    return etf_csv


try:
    scraped_etfs = scrape_investengine_etfs()
except Exception as exc:
    if INPUT_CSV.exists():
        print(f'Scrape failed, so the existing Drive CSV will be used: {exc}')
        scraped_etfs = pd.read_csv(INPUT_CSV)
    else:
        raise

print(f'Saved {len(scraped_etfs)} ETFs to {INPUT_CSV}')
print(f'Scraped at {datetime.now(timezone.utc).isoformat()}')
display(scraped_etfs.head(20))


## 6. Load ETF input CSV

In [ ]:
etfs = pd.read_csv(INPUT_CSV)
etfs.columns = etfs.columns.str.strip().str.lower()

required_columns = {'ticker', 'name'}
missing_columns = required_columns - set(etfs.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

etfs = etfs[['ticker', 'name']].dropna().drop_duplicates()
etfs['ticker'] = etfs['ticker'].astype(str).str.strip().str.upper()
etfs['name'] = etfs['name'].astype(str).str.strip()

print(f'Loaded {len(etfs)} ETFs')
display(etfs.head())

## 7. Resolve Yahoo Finance symbols

Resolution tries the raw ticker first, then these suffixes: `""`, `.L`, `.DE`, `.MI`, `.AS`, `.PA`, `.SW`, `.IR`, `.TO`, `.AX`. Results are cached in JSON so future notebook runs do not repeat successful lookups.

This step can still take time for a large ETF universe because each unresolved ticker may need several Yahoo Finance probes. The cell now prints progress and uses a per-request timeout so a single bad Yahoo request should not stall the notebook indefinitely.


In [ ]:
YFINANCE_REQUEST_TIMEOUT = 15


def load_symbol_cache(path):
    if path.exists():
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}


def save_symbol_cache(cache, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(cache, f, indent=2, sort_keys=True)


def candidate_symbols(ticker):
    return [f'{ticker}{suffix}' for suffix in YAHOO_SUFFIXES]


def close_series(data):
    close = data['Close']
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    return close.dropna()


def has_recent_prices(symbol):
    prices = yf.download(
        symbol,
        period='1mo',
        interval='1d',
        progress=False,
        auto_adjust=False,
        threads=False,
        timeout=YFINANCE_REQUEST_TIMEOUT,
    )
    return not prices.empty and 'Close' in prices.columns and close_series(prices).size > 0


def resolve_symbol(ticker, cache):
    if ticker in cache:
        return cache[ticker]

    for symbol in candidate_symbols(ticker):
        try:
            if has_recent_prices(symbol):
                cache[ticker] = symbol
                save_symbol_cache(cache, SYMBOL_CACHE_JSON)
                return symbol
        except Exception as exc:
            print(f'Skipping {symbol}: {exc}')

    cache[ticker] = None
    save_symbol_cache(cache, SYMBOL_CACHE_JSON)
    return None


symbol_cache = load_symbol_cache(SYMBOL_CACHE_JSON)
resolved_symbols = []
for index, ticker in enumerate(etfs['ticker'], start=1):
    if index == 1 or index % 10 == 0 or ticker not in symbol_cache:
        print(f'Resolving Yahoo symbol {index}/{len(etfs)}: {ticker}', flush=True)
    resolved_symbols.append(resolve_symbol(ticker, symbol_cache))

etfs['yahoo_symbol'] = resolved_symbols
unresolved = etfs[etfs['yahoo_symbol'].isna()]

if not unresolved.empty:
    print('Unresolved tickers will be skipped:')
    display(unresolved)

etfs = etfs.dropna(subset=['yahoo_symbol']).reset_index(drop=True)
print(f'Resolved {len(etfs)} ETFs')
display(etfs)


## 8. Download 5 years of daily close prices

Prices are saved in long format as `date, ticker, name, close`. Downloads now use a per-request timeout and print progress for each ETF.


In [ ]:
def download_close_prices(row):
    data = yf.download(
        row['yahoo_symbol'],
        period='5y',
        interval='1d',
        progress=False,
        auto_adjust=False,
        threads=False,
        timeout=YFINANCE_REQUEST_TIMEOUT,
    )
    if data.empty or 'Close' not in data.columns:
        return pd.DataFrame(columns=['date', 'ticker', 'name', 'close'])

    out = close_series(data).reset_index()
    out.columns = ['date', 'close']
    out['ticker'] = row['ticker']
    out['name'] = row['name']
    return out[['date', 'ticker', 'name', 'close']]


price_frames = []
for index, row in etfs.iterrows():
    print(f"Downloading {index + 1}/{len(etfs)}: {row['ticker']} from {row['yahoo_symbol']}", flush=True)
    try:
        price_frames.append(download_close_prices(row))
    except Exception as exc:
        print(f"Skipping {row['ticker']} after download failure: {exc}")

if not price_frames:
    raise ValueError('No price downloads were attempted because no Yahoo symbols were resolved.')

prices_long = pd.concat(price_frames, ignore_index=True)
if prices_long.empty:
    raise ValueError('No price data was downloaded. Check Yahoo Finance access or symbol resolution.')

prices_long['date'] = pd.to_datetime(prices_long['date']).dt.date
prices_long = prices_long.sort_values(['ticker', 'date']).reset_index(drop=True)
prices_long.to_csv(PRICES_CSV, index=False)

print(f'Saved {len(prices_long):,} rows to {PRICES_CSV}')
display(prices_long.head())


## 9. Prepare returns

In [ ]:
prices_wide = prices_long.pivot(index='date', columns='ticker', values='close').sort_index()
prices_wide = prices_wide.dropna(axis=1, thresh=int(len(prices_wide) * 0.8))
prices_wide = prices_wide.ffill().dropna()

log_returns = np.log(prices_wide / prices_wide.shift(1)).dropna()

if log_returns.empty:
    raise ValueError('No returns available after downloading prices.')

print(f'Return matrix shape: {log_returns.shape}')
display(log_returns.head())

## 10. Risk metrics, PCA covariance, and optimisation functions

In [ ]:
name_map = dict(zip(etfs['ticker'], etfs['name']))


def compute_risk_metrics(returns, window_name):
    annual_return = returns.mean() * TRADING_DAYS
    volatility = returns.std() * np.sqrt(TRADING_DAYS)
    sharpe_ratio = (annual_return - RISK_FREE_RATE) / volatility

    return pd.DataFrame({
        'window': window_name,
        'ticker': annual_return.index,
        'name': [name_map.get(ticker, ticker) for ticker in annual_return.index],
        'annual_return': annual_return.values,
        'volatility': volatility.values,
        'sharpe_ratio': sharpe_ratio.values,
    })


def pca_denoised_covariance(returns, n_components):
    cov = returns.cov().values * TRADING_DAYS
    tickers = list(returns.columns)
    n_assets = len(tickers)
    n_components = max(1, min(n_components, n_assets))

    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    denoised_eigenvalues = eigenvalues.copy()
    if n_components < n_assets:
        denoised_eigenvalues[n_components:] = eigenvalues[n_components:].mean()

    denoised_cov = eigenvectors @ np.diag(denoised_eigenvalues) @ eigenvectors.T
    denoised_cov = (denoised_cov + denoised_cov.T) / 2

    explained = pd.DataFrame({
        'component': np.arange(1, n_assets + 1),
        'explained_variance': eigenvalues,
        'explained_variance_ratio': eigenvalues / eigenvalues.sum(),
        'kept': np.arange(1, n_assets + 1) <= n_components,
    })

    loadings = pd.DataFrame(
        eigenvectors[:, :n_components],
        index=tickers,
        columns=[f'PC{i}' for i in range(1, n_components + 1)],
    ).reset_index().rename(columns={'index': 'ticker'})
    loadings['name'] = loadings['ticker'].map(name_map)

    return denoised_cov, explained, loadings


def portfolio_stats(weights, annual_returns, cov):
    portfolio_return = float(weights @ annual_returns)
    portfolio_volatility = float(np.sqrt(weights @ cov @ weights))
    portfolio_sharpe = (portfolio_return - RISK_FREE_RATE) / portfolio_volatility
    return portfolio_return, portfolio_volatility, portfolio_sharpe


def optimise_best_sharpe(returns, cov):
    tickers = list(returns.columns)
    n_assets = len(tickers)
    if n_assets * MAX_WEIGHT < 1:
        raise ValueError('MAX_WEIGHT is too low for the number of assets.')

    annual_returns = (returns.mean() * TRADING_DAYS).values
    bounds = [(0, MAX_WEIGHT)] * n_assets
    initial_weights = np.ones(n_assets) / n_assets
    target_returns = np.linspace(annual_returns.min(), annual_returns.max(), TARGET_RETURN_STEPS)
    best = None

    for target_return in target_returns:
        constraints = [
            {'type': 'eq', 'fun': lambda w: w.sum() - 1},
            {'type': 'eq', 'fun': lambda w, target=target_return: w @ annual_returns - target},
        ]
        result = minimize(
            lambda w: w @ cov @ w,
            initial_weights,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints,
            options={'maxiter': 1000, 'ftol': 1e-12},
        )
        if not result.success:
            continue

        ret, vol, sharpe = portfolio_stats(result.x, annual_returns, cov)
        if best is None or sharpe > best['sharpe_ratio']:
            best = {
                'weights': result.x,
                'annual_return': ret,
                'volatility': vol,
                'sharpe_ratio': sharpe,
            }

    if best is None:
        raise ValueError('No feasible portfolio found.')

    weights = pd.DataFrame({
        'ticker': tickers,
        'name': [name_map.get(ticker, ticker) for ticker in tickers],
        'weight': best['weights'],
    })
    return weights, best

## 11. Run analysis for both windows

In [ ]:
windows = {
    'full_5y': log_returns,
    'last_252d': log_returns.tail(TRADING_DAYS),
}

risk_metrics_all = []
explained_all = []
loadings_all = []
portfolio_all = []
portfolio_stats_all = []

for window_name, window_returns in windows.items():
    window_returns = window_returns.dropna(axis=1)
    print(f'Analysing {window_name}: {window_returns.shape[0]} days, {window_returns.shape[1]} ETFs')

    risk_metrics_all.append(compute_risk_metrics(window_returns, window_name))

    cov, explained, loadings = pca_denoised_covariance(window_returns, PCA_COMPONENTS)
    explained.insert(0, 'window', window_name)
    loadings.insert(0, 'window', window_name)
    explained_all.append(explained)
    loadings_all.append(loadings)

    weights, stats = optimise_best_sharpe(window_returns, cov)
    weights.insert(0, 'window', window_name)
    portfolio_all.append(weights)

    portfolio_stats_all.append({
        'window': window_name,
        'annual_return': stats['annual_return'],
        'volatility': stats['volatility'],
        'sharpe_ratio': stats['sharpe_ratio'],
        'max_weight': MAX_WEIGHT,
        'pca_components': min(PCA_COMPONENTS, window_returns.shape[1]),
    })

risk_metrics = pd.concat(risk_metrics_all, ignore_index=True)
pca_explained_variance = pd.concat(explained_all, ignore_index=True)
pca_loadings = pd.concat(loadings_all, ignore_index=True)
optimal_portfolio = pd.concat(portfolio_all, ignore_index=True)
optimal_portfolio_stats = pd.DataFrame(portfolio_stats_all)

## 12. Save outputs

In [ ]:
outputs = {
    'risk_metrics.csv': risk_metrics,
    'pca_explained_variance.csv': pca_explained_variance,
    'pca_loadings.csv': pca_loadings,
    'optimal_portfolio.csv': optimal_portfolio,
    'optimal_portfolio_stats.csv': optimal_portfolio_stats,
}

for filename, frame in outputs.items():
    path = PROJECT_DIR / filename
    frame.to_csv(path, index=False)
    print(f'Saved {path}')

print('Done.')

## 13. Review results

In [ ]:
display(optimal_portfolio_stats)
display(optimal_portfolio.sort_values(['window', 'weight'], ascending=[True, False]))
display(risk_metrics.sort_values(['window', 'sharpe_ratio'], ascending=[True, False]).head(20))